# vLLM_Batch — Batch Inference via Delta Queue

Processes PDF documents from a Delta table queue using vLLM for GPU-accelerated inference. Each pending request is decoded, rasterised page-by-page, and parsed into markdown via the MinerU2.5 vision-language model.

**Pattern**: Triggered Databricks Job — enqueue PDFs to Delta, run this notebook as a one-shot job, results written back to the same table.

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | g5.2xlarge or equivalent (1× NVIDIA A10G, 24 GB VRAM) |
| Runtime | `15.4.x-gpu-ml-scala2.12` (MLR 15.4 GPU) |
| Workers | 0 (single-node, `spark.master = local[*, 4]`) |
| Libraries | **None** — all packages installed via `%pip` below |

> **Important**: vLLM must NOT be installed as a cluster library. The MLR pre-installed PyTorch has custom CUDA bindings — installing vLLM at the cluster level causes ABI conflicts and kernel crashes.

### Prerequisites
- `setup/00_download_model` — model downloaded to Unity Catalog Volume
- `setup/01_prepare_test_cases` — test PDFs generated (for testing)

### Install Dependencies

| Package | Why |
|---------|-----|
| `vllm==0.7.3` | GPU inference engine (pinned — 0.8.x crashes on MLR 15.4) |
| `openai>=1.50` | Required by vLLM's OpenAI-compatible API internals |
| `transformers>=4.45,<5` | Model tokenizer/config loading (v5 removes `all_special_tokens_extended`) |
| `pymupdf` | PDF → PNG rasterisation (`fitz` module) |

In [ ]:
%pip install "vllm==0.7.3" "openai>=1.50" "transformers>=4.45,<5" pymupdf

In [ ]:
dbutils.library.restartPython()

### Configuration

Reads `config.yaml` from the project root. All notebooks share this single config file — no hardcoded catalog, schema, or volume paths.

In [ ]:
import yaml, os

# Resolve project root: __file__ exists locally, dbutils on Databricks.
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

cfg         = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG     = cfg["catalog"]
SCHEMA      = cfg["schema"]
MODEL_PATH  = f"/Volumes/{CATALOG}/{SCHEMA}/{cfg['volume']}/{cfg['model_subpath']}"
QUEUE_TABLE = f"{CATALOG}.{SCHEMA}.{cfg['queue_table']}"

# Standard prompt sent to the model for every page image
PROMPT      = "Extract all text, tables, and structure from this page. Output as clean markdown."

print(f"MODEL_PATH  : {MODEL_PATH}")
print(f"QUEUE_TABLE : {QUEUE_TABLE}")

### Queue Table

The Delta queue table stores PDF processing requests. Each row represents one document:
- **pending** → waiting to be picked up
- **processing** → currently being parsed
- **done** → markdown output written
- **error** → processing failed (see `error` column)

In [ ]:
import base64
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Create the queue table if it doesn't exist.
# Using Delta for ACID updates (pending → processing → done).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUEUE_TABLE} (
    request_id STRING NOT NULL, pdf_base64 STRING, status STRING,
    markdown STRING, char_count LONG, error STRING,
    created_at TIMESTAMP, updated_at TIMESTAMP
) USING DELTA
""")
print(f"Queue table ready: {QUEUE_TABLE}")

### Load vLLM Model

Two things happen here:
1. **Monkey-patch rope_scaling**: The MinerU2.5 `config.json` has conflicting `rope_type=default` vs `type=mrope`. vLLM 0.7.3 rejects this mismatch, so we patch the validator to reconcile them.
2. **Load the model**: vLLM's `LLM` class handles GPU allocation, KV-cache setup, and continuous batching. `limit_mm_per_prompt={"image": 1}` restricts to one image per request to stay within 24 GB VRAM.

In [ ]:
# --- Monkey-patch: fix rope_scaling conflict in model config ---
# The MinerU2.5 config.json contains:
#   "rope_scaling": {"rope_type": "default", "type": "mrope"}
# vLLM 0.7.3 raises ValueError when rope_type != type.
# Fix: override rope_type with the value of type before validation.
import vllm.transformers_utils.config as _vtc
_orig_patch = _vtc.patch_rope_scaling_dict
def _fixed_patch(rope_scaling):
    if "rope_type" in rope_scaling and "type" in rope_scaling:
        if rope_scaling["rope_type"] != rope_scaling["type"]:
            rope_scaling["rope_type"] = rope_scaling["type"]
    _orig_patch(rope_scaling)
_vtc.patch_rope_scaling_dict = _fixed_patch
print("Patched vLLM rope_scaling validator")

from vllm import LLM, SamplingParams

print(f"Loading model from {MODEL_PATH} ...")
llm = LLM(
    model=MODEL_PATH,
    dtype="bfloat16",           # Half-precision to fit in 24 GB VRAM
    max_model_len=8192,         # Max context window (tokens)
    trust_remote_code=True,     # Required for Qwen2VL architecture
    limit_mm_per_prompt={"image": 1},  # One image per request (GPU memory constraint)
)
print("vLLM model loaded")

### Helper Functions

- **`rasterize_pdf`**: Converts PDF bytes to a list of PNG images (one per page) at 150 DPI using PyMuPDF.
- **`parse_page`**: Sends a single page image to vLLM as a base64-encoded PNG and returns the model's markdown output.

In [ ]:
import fitz  # PyMuPDF

def rasterize_pdf(pdf_bytes: bytes, dpi: int = 150) -> list:
    """Convert PDF bytes to a list of PNG byte arrays, one per page."""
    doc   = fitz.open(stream=pdf_bytes, filetype="pdf")
    scale = dpi / 72.0                          # PDF points → pixels
    mat   = fitz.Matrix(scale, scale)
    pages = [page.get_pixmap(matrix=mat).tobytes("png") for page in doc]
    doc.close()
    return pages

def parse_page(img_bytes: bytes, sampling: SamplingParams) -> str:
    """Send a single page image to vLLM and return the extracted markdown."""
    img_b64  = base64.b64encode(img_bytes).decode()
    messages = [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
        {"type": "text", "text": PROMPT},
    ]}]
    # vLLM's chat API mirrors the OpenAI chat completions format
    outputs  = llm.chat(messages=messages, sampling_params=sampling)
    return outputs[0].outputs[0].text

### Process Queue

Reads all `pending` rows from the queue table and processes them sequentially:
1. Mark row as `processing`
2. Decode base64 PDF → rasterise to PNG pages
3. Run each page through vLLM → collect markdown
4. Join multi-page results with `---` separators
5. Update row to `done` (or `error` if an exception occurs)

In [ ]:
dt       = DeltaTable.forName(spark, QUEUE_TABLE)
pending  = spark.table(QUEUE_TABLE).filter(F.col("status") == "pending").collect()
sampling = SamplingParams(temperature=0.0, max_tokens=2048)  # Deterministic output
print(f"Found {len(pending)} pending record(s)")

for row in pending:
    rid = row.request_id
    print(f"\nProcessing {rid} ...")

    # Mark as processing (prevents duplicate pickup if job is re-triggered)
    dt.update(
        condition=F.col("request_id") == rid,
        set={"status": F.lit("processing"), "updated_at": F.current_timestamp()},
    )
    try:
        # Decode PDF and rasterise to per-page PNGs
        pdf_bytes  = base64.b64decode(row.pdf_base64)
        page_imgs  = rasterize_pdf(pdf_bytes)
        print(f"  Rasterized {len(page_imgs)} page(s)")

        # Run inference on each page sequentially
        page_texts = [parse_page(img, sampling) for img in page_imgs]

        # Join multi-page output with horizontal rule separators
        markdown   = "\n\n---\n\n".join(page_texts) if len(page_texts) > 1 else page_texts[0]

        # Write result back to queue table
        dt.update(
            condition=F.col("request_id") == rid,
            set={
                "status": F.lit("done"), "markdown": F.lit(markdown),
                "char_count": F.lit(len(markdown)), "error": F.lit(""),
                "updated_at": F.current_timestamp(),
            },
        )
        print(f"  Done: {len(markdown)} chars")
    except Exception as e:
        import traceback
        # Record error but don't stop processing other rows
        dt.update(
            condition=F.col("request_id") == rid,
            set={
                "status": F.lit("error"), "markdown": F.lit(""),
                "char_count": F.lit(0), "error": F.lit(str(e)),
                "updated_at": F.current_timestamp(),
            },
        )
        print(f"  Error: {e}")
        traceback.print_exc()

print(f"\nBatch complete — processed {len(pending)} record(s).")